# Homework Notebook: Convert SST `events.tsv` Files To FSL 3-Column EV Files

Audience:
- Neuroimaging class homework on Neurodesk.

What this notebook does:
- Creates or reuses a local `g25/` workspace without any `git clone` or `git pull`.
- Downloads the approved SST `events.tsv` files from OpenNeuro `ds007486` version `1.0.0`.
- Converts those BIDS event files into FEAT-ready 3-column EV files for all approved subjects.
- Shows the generated outputs inline so the notebook itself is the evidence that the workflow ran.

Important assumption:
- This notebook uses the current Jupyter working directory.
- If the current working directory is already inside a `g25/` tree, the notebook reuses that `g25/` folder.
- Otherwise, it creates a new `./g25` folder in the current working directory.


## Outline

1. Set up a portable `g25/` workspace relative to the current working directory.
2. Download the approved SST `events.tsv` files from OpenNeuro.
3. Convert those BIDS event files into FSL 3-column EV files.
4. Inspect the run summary, EV manifest, and representative EV tables inline.


In [1]:
from __future__ import annotations

import csv
import importlib.util
import json
import re
import ssl
import urllib.request
from pathlib import Path

required = {
    'pandas': 'pandas',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise RuntimeError(
        'This notebook expects the current Jupyter kernel to already provide: ' + ', '.join(missing)
    )

import pandas as pd
from IPython.display import Markdown, display

OPENNEURO_API = 'https://openneuro.org/crn/graphql'
DATASET_ID = 'ds007486'
SNAPSHOT_TAG = '1.0.0'
APPROVED_SUBJECTS = ['sub-10562', 'sub-10665', 'sub-11028', 'sub-11039', 'sub-11450']
SST_RUNS = [1, 2, 3]
ROOT_METADATA_FILES = [
    'README',
    'dataset_description.json',
    'participants.tsv',
    'participants.json',
    'task-SST_bold.json',
]
CTX = ssl._create_unverified_context()


def find_or_create_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if candidate.name == 'g25':
            return candidate
    return start / 'g25'


def gql(query: str) -> dict:
    payload = json.dumps({'query': query}).encode('utf-8')
    req = urllib.request.Request(
        OPENNEURO_API,
        data=payload,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, context=CTX) as resp:
        return json.load(resp)


def snapshot_files(tree: str | None = None) -> list[dict]:
    if tree:
        query = (
            f'query {{ snapshot(datasetId: "{DATASET_ID}", tag: "{SNAPSHOT_TAG}") '
            f'{{ files(tree: "{tree}") {{ filename directory key size urls }} }} }}'
        )
    else:
        query = (
            f'query {{ snapshot(datasetId: "{DATASET_ID}", tag: "{SNAPSHOT_TAG}") '
            f'{{ files {{ filename directory key size urls }} }} }}'
        )
    return gql(query)['data']['snapshot']['files']


def subject_tree_keys(root_rows: list[dict], subjects: list[str]) -> dict[str, str]:
    return {row['filename']: row['key'] for row in root_rows if row['directory'] and row['filename'] in subjects}


def child_directory_key(parent_key: str, dirname: str) -> str:
    rows = snapshot_files(parent_key)
    for row in rows:
        if row['directory'] and row['filename'] == dirname:
            return row['key']
    raise FileNotFoundError(f'Missing child directory {dirname} under tree {parent_key}')


def download_url(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    with urllib.request.urlopen(url, context=CTX) as resp:
        destination.write_bytes(resp.read())
    if destination.stat().st_size == 0:
        raise RuntimeError(f'Downloaded empty file for {destination.name}')
    return destination


def ensure_root_metadata(root_rows: list[dict], bids_root: Path) -> None:
    url_map = {row['filename']: row['urls'][0] for row in root_rows if row.get('urls')}
    for filename in ROOT_METADATA_FILES:
        if filename in url_map:
            download_url(url_map[filename], bids_root / filename)


def normalize_event_label(label: str) -> str:
    cleaned = re.sub(r'[^0-9A-Za-z]+', '_', label.strip()).strip('_').lower()
    return cleaned or 'unknown'


def convert_events_to_evs(events_path: Path, ev_dir: Path) -> list[dict]:
    ev_dir.mkdir(parents=True, exist_ok=True)
    with events_path.open(newline='') as src:
        reader = csv.DictReader(src, delimiter='	')
        rows = list(reader)
    if not rows:
        raise RuntimeError(f'No event rows found in {events_path}')

    grouped: dict[str, list[tuple[float, float, float]]] = {}
    for row in rows:
        trial_type = (row.get('trial_type') or '').strip()
        if not trial_type:
            continue
        onset = float(row['onset'])
        duration = float(row['duration'])
        grouped.setdefault(normalize_event_label(trial_type), []).append((onset, duration, 1.0))

    manifest_rows: list[dict] = []
    run_id = int(events_path.name.split('_events.tsv')[0].split('_run-')[-1])
    for event_label, event_rows in sorted(grouped.items()):
        out_path = ev_dir / f'run-{run_id}_{event_label}.txt'
        with out_path.open('w', newline='') as dst:
            writer = csv.writer(dst, delimiter='\t', lineterminator='\n')
            for onset, duration, amplitude in sorted(event_rows, key=lambda item: item[0]):
                writer.writerow([f'{onset:.6f}', f'{duration:.6f}', f'{amplitude:.1f}'])
        manifest_rows.append({
            'run': run_id,
            'event': event_label,
            'rows': len(event_rows),
            'path': str(out_path),
        })
    return manifest_rows


WORKSPACE_ROOT = Path.cwd().resolve()
PROJECT_ROOT = find_or_create_project_root(WORKSPACE_ROOT)
BIDS_ROOT = PROJECT_ROOT / 'bids'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'jupyter-notebook'
EV_ROOT = PROJECT_ROOT / 'derivatives' / 'fsl' / 'EVfiles'
README_PATH = PROJECT_ROOT / 'README.md'

print(f'Current working directory: {WORKSPACE_ROOT}')
print(f'g25 project root: {PROJECT_ROOT}')
print(f'Approved subjects: {APPROVED_SUBJECTS}')
print(f'SST runs: {SST_RUNS}')


Current working directory: /Users/kwakufinest/Documents/New project/group-3/g25/output/jupyter-notebook
g25 project root: /Users/kwakufinest/Documents/New project/group-3/g25
Approved subjects: ['sub-10562', 'sub-10665', 'sub-11028', 'sub-11039', 'sub-11450']
SST runs: [1, 2, 3]


## Step 1 - Create the local `g25/` workspace

Purpose:
- The homework should run without assuming a pre-existing clone.
- This step creates the folder structure the rest of the notebook will use.

What the code is doing:
- Reuse an existing ancestor `g25/` folder if the notebook is already being run inside one.
- Otherwise create a new `g25/` folder under the current working directory.
- Create the minimal `bids/`, `derivatives/`, and `output/` folders needed for this homework.
- Write a short local `README.md` so the workspace is self-describing.

QC checkpoint:
- The table below should show `exists=True` for every required directory.


In [2]:
for path in [BIDS_ROOT, EV_ROOT, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not README_PATH.exists():
    README_PATH.write_text(
        '# g25 Homework Workspace\n\n'
        + f'Dataset: {DATASET_ID} (snapshot {SNAPSHOT_TAG})\n\n'
        + 'Purpose:\n'
        + '- download approved SST events.tsv files\n'
        + '- convert them to FSL 3-column EV files\n'
    )

workspace_table = pd.DataFrame([
    {'artifact': 'project_root', 'path': str(PROJECT_ROOT), 'exists': PROJECT_ROOT.exists()},
    {'artifact': 'bids_root', 'path': str(BIDS_ROOT), 'exists': BIDS_ROOT.exists()},
    {'artifact': 'ev_root', 'path': str(EV_ROOT), 'exists': EV_ROOT.exists()},
    {'artifact': 'output_dir', 'path': str(OUTPUT_DIR), 'exists': OUTPUT_DIR.exists()},
    {'artifact': 'readme', 'path': str(README_PATH), 'exists': README_PATH.exists()},
])
display(workspace_table)


,artifact,path,exists
0,project_root,/Users/kwakufinest/Documents/New project/group...,True
1,bids_root,/Users/kwakufinest/Documents/New project/group...,True
2,ev_root,/Users/kwakufinest/Documents/New project/group...,True
3,output_dir,/Users/kwakufinest/Documents/New project/group...,True
4,readme,/Users/kwakufinest/Documents/New project/group...,True


## Step 2 - Download the approved SST `events.tsv` files and convert them to EV files

Purpose:
- This step performs the actual homework workflow.

What the code is doing:
- Download dataset-level metadata into `g25/bids/`.
- Download the approved SST `events.tsv` files for every approved subject and SST run.
- Convert each `trial_type` category into its own FSL 3-column EV file with columns `onset`, `duration`, and `amplitude`.
- Write two compact outputs into `g25/output/jupyter-notebook/`:
  - `g25-sst-all-subjects-run-summary.csv`
  - `g25-sst-all-subjects-ev-manifest.csv`

QC checkpoint:
- The run summary should show all approved subjects and SST runs.
- Every EV file should contain exactly three columns.


In [3]:
root_rows = snapshot_files()
ensure_root_metadata(root_rows, BIDS_ROOT)
subject_keys = subject_tree_keys(root_rows, APPROVED_SUBJECTS)

manifest_rows = []
run_summary_rows = []

for subject in APPROVED_SUBJECTS:
    func_key = child_directory_key(subject_keys[subject], 'func')
    func_rows = snapshot_files(func_key)
    url_map = {row['filename']: row['urls'][0] for row in func_rows if row.get('urls')}

    for run in SST_RUNS:
        events_name = f'{subject}_task-SST_run-{run}_events.tsv'
        events_path = BIDS_ROOT / subject / 'func' / events_name
        if events_name not in url_map:
            run_summary_rows.append({
                'subject': subject,
                'run': run,
                'status': 'missing-events-tsv',
                'n_ev_files': 0,
            })
            continue

        download_url(url_map[events_name], events_path)
        run_ev_dir = EV_ROOT / subject / 'SST'
        run_manifest = convert_events_to_evs(events_path, run_ev_dir)
        run_summary_rows.append({
            'subject': subject,
            'run': run,
            'status': 'ok',
            'n_ev_files': len(run_manifest),
        })
        for row in run_manifest:
            manifest_rows.append({
                'subject': subject,
                **row,
            })

RUN_SUMMARY_PATH = OUTPUT_DIR / 'g25-sst-all-subjects-run-summary.csv'
EV_MANIFEST_PATH = OUTPUT_DIR / 'g25-sst-all-subjects-ev-manifest.csv'
WORKSPACE_SUMMARY_PATH = OUTPUT_DIR / 'g25-homework-workspace-summary.json'

run_summary = pd.DataFrame(run_summary_rows).sort_values(['subject', 'run']).reset_index(drop=True)
ev_manifest = pd.DataFrame(manifest_rows).sort_values(['subject', 'run', 'event']).reset_index(drop=True)

run_summary.to_csv(RUN_SUMMARY_PATH, index=False)
ev_manifest.to_csv(EV_MANIFEST_PATH, index=False)
WORKSPACE_SUMMARY_PATH.write_text(json.dumps({
    'workspace_root': str(WORKSPACE_ROOT),
    'project_root': str(PROJECT_ROOT),
    'run_summary_path': str(RUN_SUMMARY_PATH),
    'ev_manifest_path': str(EV_MANIFEST_PATH),
    'approved_subjects': APPROVED_SUBJECTS,
    'sst_runs': SST_RUNS,
}, indent=2))

print(f'Run summary: {RUN_SUMMARY_PATH}')
print(f'EV manifest: {EV_MANIFEST_PATH}')
print(f'Total EV files: {len(ev_manifest)}')

display(run_summary)
assert set(run_summary['subject']) == set(APPROVED_SUBJECTS)
assert set(run_summary['run']) == set(SST_RUNS)
assert not ev_manifest.empty
assert all(pd.read_csv(Path(path), sep='	', header=None).shape[1] == 3 for path in ev_manifest['path'])


Run summary: /Users/kwakufinest/Documents/New project/group-3/g25/output/jupyter-notebook/g25-sst-all-subjects-run-summary.csv
EV manifest: /Users/kwakufinest/Documents/New project/group-3/g25/output/jupyter-notebook/g25-sst-all-subjects-ev-manifest.csv
Total EV files: 65


,subject,run,status,n_ev_files
0,sub-10562,1,ok,6
1,sub-10562,2,ok,5
2,sub-10562,3,ok,6
3,sub-10665,1,ok,4
4,sub-10665,2,ok,4
5,sub-10665,3,ok,4
6,sub-11028,1,ok,4
7,sub-11028,2,ok,4
8,sub-11028,3,ok,4
9,sub-11039,1,ok,4


## Step 3 - Inspect the generated EV outputs inline

Purpose:
- The notebook itself should show that the conversion worked.

What the code is doing:
- Read the summary and manifest files back from disk.
- Preview the first part of the EV manifest.
- Show representative EV file contents inline.
- Print a small relative-path tree so it is obvious where the files live.

QC checkpoint:
- The EV previews should show three columns: onset, duration, amplitude.
- The relative tree should show `g25/derivatives/fsl/EVfiles/sub-*/SST/` paths.


In [4]:
run_summary = pd.read_csv(RUN_SUMMARY_PATH)
ev_manifest = pd.read_csv(EV_MANIFEST_PATH)
workspace_summary = json.loads(WORKSPACE_SUMMARY_PATH.read_text())

print('Workspace summary:')
print(json.dumps(workspace_summary, indent=2))

display(Markdown('**Run Summary**'))
display(run_summary)

display(Markdown('**EV Manifest Preview**'))
display(ev_manifest.head(20))

preview_paths = [Path(path) for path in ev_manifest['path'].head(6)]
preview_tables = []
for ev_path in preview_paths:
    table = pd.read_csv(ev_path, sep='	', header=None, names=['onset', 'duration', 'amplitude'])
    table.insert(0, 'file', ev_path.name)
    preview_tables.append(table.head())

display(Markdown('**Representative EV File Preview**'))
display(pd.concat(preview_tables, ignore_index=True))

relative_tree = []
for subject in APPROVED_SUBJECTS:
    subject_ev_dir = EV_ROOT / subject / 'SST'
    relative_tree.append(str(subject_ev_dir.relative_to(PROJECT_ROOT)))
    for ev_path in sorted(subject_ev_dir.glob('run-*_*.txt'))[:4]:
        relative_tree.append('  ' + str(ev_path.relative_to(PROJECT_ROOT)))

print('Relative EV tree preview:')
print('\n'.join(relative_tree))


Workspace summary:
{
  "workspace_root": "/Users/kwakufinest/Documents/New project/group-3/g25/output/jupyter-notebook",
  "project_root": "/Users/kwakufinest/Documents/New project/group-3/g25",
  "run_summary_path": "/Users/kwakufinest/Documents/New project/group-3/g25/output/jupyter-notebook/g25-sst-all-subjects-run-summary.csv",
  "ev_manifest_path": "/Users/kwakufinest/Documents/New project/group-3/g25/output/jupyter-notebook/g25-sst-all-subjects-ev-manifest.csv",
  "approved_subjects": [
    "sub-10562",
    "sub-10665",
    "sub-11028",
    "sub-11039",
    "sub-11450"
  ],
  "sst_runs": [
    1,
    2,
    3
  ]
}


**Run Summary**

,subject,run,status,n_ev_files
0,sub-10562,1,ok,6
1,sub-10562,2,ok,5
2,sub-10562,3,ok,6
3,sub-10665,1,ok,4
4,sub-10665,2,ok,4
5,sub-10665,3,ok,4
6,sub-11028,1,ok,4
7,sub-11028,2,ok,4
8,sub-11028,3,ok,4
9,sub-11039,1,ok,4


**EV Manifest Preview**

,subject,run,event,rows,path
0,sub-10562,1,fixation,74,/Users/kwakufinest/Documents/New project/group...
1,sub-10562,1,go_correct,33,/Users/kwakufinest/Documents/New project/group...
2,sub-10562,1,go_incorrect,2,/Users/kwakufinest/Documents/New project/group...
3,sub-10562,1,go_miss,23,/Users/kwakufinest/Documents/New project/group...
4,sub-10562,1,stop_failure,3,/Users/kwakufinest/Documents/New project/group...
5,sub-10562,1,stop_success,13,/Users/kwakufinest/Documents/New project/group...
6,sub-10562,2,fixation,74,/Users/kwakufinest/Documents/New project/group...
7,sub-10562,2,go_correct,28,/Users/kwakufinest/Documents/New project/group...
8,sub-10562,2,go_miss,24,/Users/kwakufinest/Documents/New project/group...
9,sub-10562,2,stop_failure,4,/Users/kwakufinest/Documents/New project/group...


**Representative EV File Preview**

,file,onset,duration,amplitude
0,run-1_fixation.txt,0.010406,1.628609,1.0
1,run-1_fixation.txt,7.289105,1.249734,1.0
2,run-1_fixation.txt,11.556126,1.399719,1.0
3,run-1_fixation.txt,18.606225,1.433120,1.0
4,run-1_fixation.txt,25.872692,1.933245,1.0
5,run-1_go_correct.txt,8.538839,0.553000,1.0
6,run-1_go_correct.txt,12.955845,0.570000,1.0
7,run-1_go_correct.txt,36.006232,0.714000,1.0
8,run-1_go_correct.txt,40.023008,0.331000,1.0
9,run-1_go_correct.txt,49.656357,0.591000,1.0


Relative EV tree preview:
derivatives/fsl/EVfiles/sub-10562/SST
  derivatives/fsl/EVfiles/sub-10562/SST/run-1_fixation.txt
  derivatives/fsl/EVfiles/sub-10562/SST/run-1_go_correct.txt
  derivatives/fsl/EVfiles/sub-10562/SST/run-1_go_incorrect.txt
  derivatives/fsl/EVfiles/sub-10562/SST/run-1_go_miss.txt
derivatives/fsl/EVfiles/sub-10665/SST
  derivatives/fsl/EVfiles/sub-10665/SST/run-1_fixation.txt
  derivatives/fsl/EVfiles/sub-10665/SST/run-1_go_correct.txt
  derivatives/fsl/EVfiles/sub-10665/SST/run-1_stop_failure.txt
  derivatives/fsl/EVfiles/sub-10665/SST/run-1_stop_success.txt
derivatives/fsl/EVfiles/sub-11028/SST
  derivatives/fsl/EVfiles/sub-11028/SST/run-1_fixation.txt
  derivatives/fsl/EVfiles/sub-11028/SST/run-1_go_correct.txt
  derivatives/fsl/EVfiles/sub-11028/SST/run-1_stop_failure.txt
  derivatives/fsl/EVfiles/sub-11028/SST/run-1_stop_success.txt
derivatives/fsl/EVfiles/sub-11039/SST
  derivatives/fsl/EVfiles/sub-11039/SST/run-1_fixation.txt
  derivatives/fsl/EVfiles/sub-

## Submission notes

This notebook leaves behind:
- `g25/bids/sub-*/func/*_events.tsv`
- `g25/derivatives/fsl/EVfiles/sub-*/SST/run-*_*.txt`
- `g25/output/jupyter-notebook/g25-sst-all-subjects-run-summary.csv`
- `g25/output/jupyter-notebook/g25-sst-all-subjects-ev-manifest.csv`

Why this is portable:
- No `git clone`
- No `git pull`
- No absolute paths baked into the logic
- The workspace root is determined from the current working directory at runtime
